In [7]:
import pandas as pd
import requests
import pandas as pd
import io
import pyarrow as pa
import pyarrow.parquet as pq
import os

# set global variables
# note: start and end date should not be hardcoded in production code, 
# but for the sake of this example, we will use them as constants
START_DATE = "2022-01-01"
END_DATE   = "2022-01-07"

In [ ]:
# fetch dataset name

resp = requests.get(
"https://data.cityofnewyork.us/api/catalog/v1",
    params={"q": "yellow taxi trip data",
"limit": 30}
)

datasets = {
    r["resource"]["name"]: r["resource"]["id"]
    for r in resp.json()["results"]
    if "yellow taxi trip data" in
r["resource"]["name"].lower() and 
("2022" in r["resource"]["name"].lower() 
 or "2023" in r["resource"]["name"].lower())
}

In [8]:
# ── config ────────────────────────────────────────────────────────────────────
DATASET_ID       = "qp3b-zxtp"          # 2022 Yellow Taxi Trip Data
BASE_URL         = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.json"
PAGE_SIZE        = 50_000
START_DATE       = "2022-01-01"
END_DATE         = "2022-01-14"
END_EXCLUSIVE    = "2022-01-15"

# ── output path ───────────────────────────────────────────────────────────────
os.makedirs("data", exist_ok=True)
output_path = f"data/yellow_tripdata_{START_DATE}_{END_DATE}.parquet"

# ── dtypes applied to every batch ─────────────────────────────────────────────
DATETIME_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]
FLOAT_COLS    = ["trip_distance", "fare_amount", "extra", "mta_tax",
                 "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount"]
INT_COLS      = ["vendorid", "passenger_count", "ratecodeid",
                 "payment_type", "pulocationid", "dolocationid"]

def cast_batch(df: pd.DataFrame) -> pd.DataFrame:
    for col in DATETIME_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    for col in FLOAT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")
    for col in INT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int16")
    return df

# ── batch fetch + stream write ────────────────────────────────────────────────
where_clause = (
    f"tpep_pickup_datetime >= '{START_DATE}T00:00:00' "
    f"AND tpep_pickup_datetime < '{END_EXCLUSIVE}T00:00:00'"
)

writer      = None
offset      = 0
total_rows  = 0

try:
    while True:
        params = {
            "$where" : where_clause,
            "$limit" : PAGE_SIZE,
            "$offset": offset,
            "$order" : "tpep_pickup_datetime",
        }
        resp = requests.get(BASE_URL, params=params, timeout=60)
        resp.raise_for_status()

        page = resp.json()
        if not page:
            break

        batch = cast_batch(pd.DataFrame(page))
        table = pa.Table.from_pandas(batch, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema, compression="snappy")

        writer.write_table(table)
        total_rows += len(batch)
        print(f"  offset={offset:>7,}  batch={len(batch):,}  total={total_rows:,}")

        # done when page is smaller than requested
        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

finally:
    if writer:
        writer.close()
        print(f"\nSaved → {output_path}  ({total_rows:,} rows)")

  offset=      0  batch=50,000  total=50,000
  offset= 50,000  batch=50,000  total=100,000
  offset=100,000  batch=50,000  total=150,000
  offset=150,000  batch=50,000  total=200,000
  offset=200,000  batch=50,000  total=250,000
  offset=250,000  batch=50,000  total=300,000
  offset=300,000  batch=50,000  total=350,000
  offset=350,000  batch=50,000  total=400,000
  offset=400,000  batch=50,000  total=450,000
  offset=450,000  batch=50,000  total=500,000
  offset=500,000  batch=50,000  total=550,000
  offset=550,000  batch=50,000  total=600,000
  offset=600,000  batch=50,000  total=650,000
  offset=650,000  batch=50,000  total=700,000
  offset=700,000  batch=50,000  total=750,000
  offset=750,000  batch=50,000  total=800,000
  offset=800,000  batch=50,000  total=850,000
  offset=850,000  batch=50,000  total=900,000
  offset=900,000  batch=50,000  total=950,000
  offset=950,000  batch=50,000  total=1,000,000
  offset=1,000,000  batch=50,000  total=1,050,000
  offset=1,050,000  batch=2,5

In [10]:
# Verify the saved file
saved = pq.read_metadata(output_path)
print(f"Row groups : {saved.num_row_groups}")
print(f"Total rows : {saved.num_rows:,}")
print(f"File size  : {os.path.getsize(output_path) / 1024**2:.1f} MB")

Row groups : 20
Total rows : 1,000,000
File size  : 20.3 MB
